# Reproducing the Results Figures

This notebook reproduces all figures presented in the **Results** section. 

In [ ]:
from pathlib import Path
import os
import platform

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import FormatStrFormatter
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)


# User configuration
DATA_ROOT = Path(os.environ.get("GRB_RESULTS_ROOT", ".")).expanduser().resolve()

output_override = os.environ.get("GRB_FIGURE_OUTPUT_DIR")
OUTPUT_DIR = (
    Path(output_override).expanduser().resolve()
    if output_override
    else DATA_ROOT / "figures"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAPPING_INPUTS = [
    ("Long Transients", DATA_ROOT / "training" / "longlow_mapping_time_stats.csv"),
    ("Short Transients", DATA_ROOT / "training" / "short_mapping_time_stats.csv"),
]

SEARCH_INPUTS = [
    ("Long 5.36° × 4.5°", DATA_ROOT / "training" / "longlow_searching_5.36x4.5_tiling.csv"),
    ("Long 2.5° × 2.5°", DATA_ROOT / "training" / "longlow_searching_2.5x2.5_tiling.csv"),
    ("Short 5.36° × 4.5°", DATA_ROOT / "training" / "short_searching_5.36x4.5_tiling.csv"),
    ("Short 2.5° × 2.5°", DATA_ROOT / "training" / "short_searching_2.5x2.5_tiling.csv"),
]

SHORT_BASELINE = (
    DATA_ROOT / "validation" / "mapping_stats" / "no-fov_short" / "emsoft_stats_nodeadline.csv"
)
LONG_BASELINE = (
    DATA_ROOT / "validation" / "mapping_stats" / "no-fov_longlow" / "emsoft_stats_nodeadline.csv"
)
LONG_UTILITY_DIR = DATA_ROOT / "validation" / "mapping_stats" / "5.36x4.5_longlow"

DEADLINES_SHORT = [10, 15, 20, 25, 30, 35, 40, 45, 50]
DEADLINES_LONG = [30, 40, 50, 60, 70, 80, 90, 100, 110]

# Paper order: short/long at the larger FoV, then short/long at the smaller FoV.
SUCCESS_CONFIGS = [
    ("fig:success_short_large_fov", "short", "5.36x4.5", DEADLINES_SHORT),
    ("fig:success_long_large_fov", "longlow", "5.36x4.5", DEADLINES_LONG),
    ("fig:success_short_small_fov", "short", "2.5x2.5", DEADLINES_SHORT),
    ("fig:success_long_small_fov", "longlow", "2.5x2.5", DEADLINES_LONG),
]

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times", "Times New Roman", "DejaVu Serif"],
    "font.size": 8,
    "axes.labelsize": 8,
    "axes.titlesize": 8,
    "legend.fontsize": 7,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "mathtext.fontset": "stix",
    "axes.unicode_minus": False,
    "savefig.bbox": "tight",
})

GENERATED_FILES = set()

print(f"Python:     {platform.python_version()}")
print(f"NumPy:      {np.__version__}")
print(f"pandas:     {pd.__version__}")
print(f"Matplotlib: {mpl.__version__}")
print(f"Data root:  {DATA_ROOT}")
print(f"Figures:    {OUTPUT_DIR}")

## Shared validation and output helpers

Each figure is saved in both paper-ready PDF and convenient 300-dpi PNG form.

In [ ]:
def read_required_csv(path, required_columns):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(
            f"Required input not found: {path}\n"
            "Edit DATA_ROOT/the path configuration, or generate the missing result."
        )
    df = pd.read_csv(path)
    missing = sorted(set(required_columns) - set(df.columns))
    if missing:
        raise ValueError(f"{path} is missing required columns: {missing}")
    if df.empty:
        raise ValueError(f"{path} contains no rows")
    return df


def detected_mask(series):
    """Convert common bool, 0/1, and text encodings to a Boolean mask."""
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False).astype(bool)
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="raise").fillna(0).ne(0)

    normalized = series.astype("string").str.strip().str.lower()
    mapping = {
        "true": True, "t": True, "yes": True, "y": True, "1": True,
        "false": False, "f": False, "no": False, "n": False, "0": False,
    }
    unknown = sorted(set(normalized.dropna()) - set(mapping))
    if unknown:
        raise ValueError(f"Unrecognized values in detected column: {unknown[:5]}")
    return normalized.map(mapping).fillna(False).astype(bool)


def numeric_array(df, column):
    values = pd.to_numeric(df[column], errors="raise").to_numpy(dtype=float)
    if not np.isfinite(values).all():
        raise ValueError(f"Column {column!r} contains NaN or infinite values")
    return values


def draw_boxplot(ax, data, labels, **kwargs):
    """Support both pre-3.9 and current Matplotlib boxplot keyword names."""
    try:
        return ax.boxplot(data, tick_labels=labels, **kwargs)
    except TypeError:
        return ax.boxplot(data, labels=labels, **kwargs)


def sampled_frame(df, max_points=10_000, seed=42):
    if len(df) <= max_points:
        return df
    return df.sample(n=max_points, replace=False, random_state=seed)


def save_and_show(fig, stem):
    pdf_path = OUTPUT_DIR / f"{stem}.pdf"
    png_path = OUTPUT_DIR / f"{stem}.png"
    fig.savefig(pdf_path)
    fig.savefig(png_path, dpi=300)
    GENERATED_FILES.update({pdf_path, png_path})
    print(f"Saved {pdf_path}")
    print(f"Saved {png_path}")
    plt.show()
    plt.close(fig)
    return pdf_path, png_path


def verify_paired_rows(reference, comparison):
    if len(reference) != len(comparison):
        raise ValueError(
            "The baseline and utility CSVs must contain the same number of rows; "
            f"found {len(reference)} and {len(comparison)}."
        )

    # If a recognizable identifier exists in both files, verify it explicitly.
    for key in ("transient_id", "event_id", "sample_id", "id"):
        if key in reference.columns and key in comparison.columns:
            if not reference[key].reset_index(drop=True).equals(
                comparison[key].reset_index(drop=True)
            ):
                raise ValueError(f"Paired rows do not align by {key!r}")
            print(f"Verified row alignment using {key!r}.")
            return
    print("No identifier column found; using the original row-order pairing assumption.")

## Figure 5 

Left: Box plots of map-generation times for 52,800 simulated TAPs on our target platform for both short and long transients. 

Right: Map-generation time vs. number of source events s = e − b. Most outliers occur when s < 350.

In [ ]:
def make_mapping_time_figure(datasets, quantile=0.95, y_clip=10):
    labels, box_data, quantiles, summaries = [], [], [], []

    for label, df in datasets:
        runtime = numeric_array(df, "mapping_time")
        q_value = float(np.quantile(runtime, quantile))
        labels.append(label)
        quantiles.append(q_value)
        box_data.append(np.clip(runtime, None, y_clip))
        summaries.append({
            "dataset": label,
            "n": len(runtime),
            "median_s": np.median(runtime),
            f"p{int(100 * quantile)}_s": q_value,
            "max_s": np.max(runtime),
        })

    palette = ["#3b7dd8", "#d84b4b", "#2ca02c", "#9467bd"][:len(datasets)]
    fig = plt.figure(
        figsize=(4.5, 1.0 + len(datasets)), constrained_layout=True
    )
    outer = fig.add_gridspec(1, 2, width_ratios=[1, 1.3])
    ax_box = fig.add_subplot(outer[0])
    inner = outer[1].subgridspec(len(datasets), 1, hspace=0.08)
    scatter_axes = []
    for i in range(len(datasets)):
        shared = scatter_axes[0] if scatter_axes else None
        scatter_axes.append(fig.add_subplot(inner[i], sharex=shared))

    bp = draw_boxplot(
        ax_box,
        box_data,
        labels,
        patch_artist=True,
        widths=0.5,
        medianprops={"color": "black", "linewidth": 1.2},
        flierprops={"marker": ".", "markersize": 2, "alpha": 0.4},
        whiskerprops={"linewidth": 0.8},
        capprops={"linewidth": 0.8},
        boxprops={"linewidth": 0.8},
    )
    for patch, color in zip(bp["boxes"], palette):
        patch.set_facecolor(color)
        patch.set_alpha(0.35)

    for i, q_value in enumerate(quantiles, start=1):
        draw_value = min(q_value, y_clip)
        ax_box.plot(
            [i - 0.25, i + 0.25], [draw_value, draw_value],
            color="tab:red", linewidth=1.2, zorder=5,
        )
        ax_box.annotate(
            f"{q_value:.3f}s", xy=(i - 0.15, draw_value),
            xytext=(0, 4), textcoords="offset points",
            fontsize=6, color="tab:red", ha="left", va="bottom",
        )

    ax_box.legend(
        handles=[Line2D([0], [0], color="tab:red", linewidth=1.2,
                        label=f"{int(100 * quantile)}th percentile")],
        loc="upper right",
    )
    ax_box.set_ylabel("Map Generation Time (s)")
    ax_box.tick_params(axis="x", rotation=30)
    for tick in ax_box.get_xticklabels():
        tick.set_horizontalalignment("right")
    ax_box.set_ylim(-0.3, y_clip)
    ticks = np.linspace(0, y_clip, 6)
    tick_labels = [f"{tick:g}" for tick in ticks]
    tick_labels[-1] = f"{y_clip:g}+"
    ax_box.set_yticks(ticks, tick_labels)

    for i, ((label, df), color, ax) in enumerate(
        zip(datasets, palette, scatter_axes)
    ):
        sample = sampled_frame(df)
        ax.scatter(
            numeric_array(sample, "n_src"),
            numeric_array(sample, "mapping_time"),
            marker=".", color=color, s=2, alpha=0.5,
            edgecolors="black", linewidths=0.1, rasterized=True,
        )
        ax.set_ylabel("Time (s)", fontsize=7)
        ax.yaxis.set_major_formatter(FormatStrFormatter("%.1f"))
        ax.text(
            0.98, 0.92, label, transform=ax.transAxes,
            ha="right", va="top", fontsize=7, color=color,
            bbox={"boxstyle": "round,pad=0.2", "facecolor": "white",
                  "edgecolor": color, "alpha": 0.7, "linewidth": 0},
        )
        if i < len(scatter_axes) - 1:
            ax.tick_params(axis="x", labelbottom=False)
    scatter_axes[-1].set_xlabel("Source Events, $s$")

    return fig, pd.DataFrame(summaries)


mapping_datasets = [
    (label, read_required_csv(path, {"mapping_time", "n_src"}))
    for label, path in MAPPING_INPUTS
]
mapping_figure, mapping_summary = make_mapping_time_figure(mapping_datasets)
display(mapping_summary.round({
    "median_s": 4, "p95_s": 4, "max_s": 4,
}))
save_and_show(mapping_figure, "mapping_time_combined")

## Figure 6 

Left: Box plots of search-planning time for short and long transients using different partner FoVs. 

Right: Search-planning time vs. number of source events. Most outliers have s < 200.

In [ ]:
def make_search_planning_figure(datasets, quantile=0.95, y_clip=3):
    labels, box_data, quantiles, summaries = [], [], [], []

    for label, df in datasets:
        runtime = numeric_array(df, "runtime")
        q_value = float(np.quantile(runtime, quantile))
        labels.append(label)
        quantiles.append(q_value)
        box_data.append(np.clip(runtime, None, y_clip))
        summaries.append({
            "dataset": label,
            "n_detected": len(runtime),
            "median_s": np.median(runtime),
            f"p{int(100 * quantile)}_s": q_value,
            "n_over_0.1_s": int(np.sum(runtime > 0.1)),
            "fraction_over_0.1_s": float(np.mean(runtime > 0.1)),
        })

    palette = ["#3b7dd8", "#d84b4b", "#2ca02c", "#9467bd"][:len(datasets)]
    fig = plt.figure(
        figsize=(4.5, 0.5 + 0.5 * len(datasets)), constrained_layout=True
    )
    outer = fig.add_gridspec(1, 2, width_ratios=[1, 1.3])
    ax_box = fig.add_subplot(outer[0])
    inner = outer[1].subgridspec(len(datasets), 1, hspace=0.08)
    scatter_axes = []
    for i in range(len(datasets)):
        shared = scatter_axes[0] if scatter_axes else None
        scatter_axes.append(fig.add_subplot(inner[i], sharex=shared))

    bp = draw_boxplot(
        ax_box,
        box_data,
        labels,
        patch_artist=True,
        widths=0.5,
        medianprops={"color": "black", "linewidth": 1.2},
        flierprops={"marker": ".", "markersize": 2, "alpha": 0.4},
        whiskerprops={"linewidth": 0.8},
        capprops={"linewidth": 0.8},
        boxprops={"linewidth": 0.8},
    )
    for patch, color in zip(bp["boxes"], palette):
        patch.set_facecolor(color)
        patch.set_alpha(0.35)

    for i, q_value in enumerate(quantiles, start=1):
        draw_value = min(q_value, y_clip)
        ax_box.plot(
            [i - 0.25, i + 0.25], [draw_value, draw_value],
            color="tab:red", linewidth=1.2, zorder=5,
        )
        ax_box.annotate(
            f"{q_value:.3f}s", xy=(i - 0.15, draw_value),
            xytext=(0, 4), textcoords="offset points",
            fontsize=6, color="tab:red", ha="left", va="bottom",
        )

    ax_box.legend(
        handles=[Line2D([0], [0], color="tab:red", linewidth=1.2,
                        label=f"{int(100 * quantile)}th percentile")],
        loc="upper right",
    )
    ax_box.set_ylabel("Search Planning Time (s)")
    ax_box.tick_params(axis="x", rotation=30)
    for tick in ax_box.get_xticklabels():
        tick.set_horizontalalignment("right")
    ax_box.set_ylim(-0.3, y_clip)
    ticks = np.linspace(0, y_clip, 4)
    tick_labels = [f"{tick:g}" for tick in ticks]
    tick_labels[-1] = f"{y_clip:g}+"
    ax_box.set_yticks(ticks, tick_labels)

    for i, ((label, df), color, ax) in enumerate(
        zip(datasets, palette, scatter_axes)
    ):
        sample = sampled_frame(df)
        ax.scatter(
            numeric_array(sample, "n_src"),
            numeric_array(sample, "runtime"),
            marker=".", color=color, s=2, alpha=0.5,
            edgecolors="black", linewidths=0.1, rasterized=True,
        )
        ax.set_ylabel("Time (s)", fontsize=7)
        ax.yaxis.set_major_formatter(FormatStrFormatter("%.1f"))
        ax.text(
            0.98, 0.92, label, transform=ax.transAxes,
            ha="right", va="top", fontsize=7, color=color,
            bbox={"boxstyle": "round,pad=0.2", "facecolor": "white",
                  "edgecolor": color, "alpha": 0.7, "linewidth": 0},
        )
        if i < len(scatter_axes) - 1:
            ax.tick_params(axis="x", labelbottom=False)
    scatter_axes[-1].set_xlabel("Source Events, $s$")

    return fig, pd.DataFrame(summaries)


search_datasets = []
for label, path in SEARCH_INPUTS:
    all_rows = read_required_csv(path, {"detected", "runtime", "n_src"})
    detected_rows = all_rows.loc[detected_mask(all_rows["detected"])].copy()
    if detected_rows.empty:
        raise ValueError(f"No detected rows remain after filtering {path}")
    print(f"{path}: {len(detected_rows)} detected / {len(all_rows)} total")
    search_datasets.append((label, detected_rows))

search_figure, search_summary = make_search_planning_figure(search_datasets)
display(search_summary.round({
    "median_s": 4, "p95_s": 4, "fraction_over_0.1_s": 4,
}))
save_and_show(search_figure, "search_planning_time_combined")

## Figure 7

Cumulative distribution of mapping errors for short and long transients. CDFs reach 1 at ∼160° error.

In [ ]:
def empirical_cdf(values):
    x = np.sort(np.asarray(values, dtype=float))
    y = np.arange(1, len(x) + 1, dtype=float) / len(x)
    return x, y


def make_error_ecdf(short_error, long_error):
    fig, ax = plt.subplots(figsize=(3.5, 2.6), constrained_layout=True)
    x_short, y_short = empirical_cdf(short_error)
    x_long, y_long = empirical_cdf(long_error)
    ax.step(x_short, y_short, where="post", label="Short")
    ax.step(x_long, y_long, where="post", linestyle="--", label="Long")
    ax.set_xlim(0, 20)
    ax.set_ylim(0, 1.01)
    ax.set_xlabel("Error (deg)")
    ax.set_ylabel("Fraction of transients")
    ax.legend()
    return fig


short_baseline_df = read_required_csv(
    SHORT_BASELINE, {"n_src", "n_bkg", "exp_angdist"}
)
long_baseline_df = read_required_csv(
    LONG_BASELINE, {"n_src", "n_bkg", "exp_angdist"}
)

short_error = numeric_array(short_baseline_df, "exp_angdist")
long_error = numeric_array(long_baseline_df, "exp_angdist")

error_ecdf_figure = make_error_ecdf(short_error, long_error)
ecdf_summary = pd.DataFrame({
    "dataset": ["Short", "Long"],
    "n": [len(short_error), len(long_error)],
    "median_error_deg": [np.median(short_error), np.median(long_error)],
    "p68_error_deg": [np.quantile(short_error, 0.68), np.quantile(long_error, 0.68)],
    "p80_error_deg": [np.quantile(short_error, 0.80), np.quantile(long_error, 0.80)],
})
display(ecdf_summary.round({
    "median_error_deg": 3, "p68_error_deg": 3, "p80_error_deg": 3,
}))
save_and_show(error_ecdf_figure, "map_err_ecdf")

## Figure 8

Dependence of mapping error on numbers of source and background events. Horizontal stripes reflect integer- valued transient lengths inferred by mapping; these stripes are more numerous and overlap for long transients.

In [ ]:
def make_error_vs_events_figure(df, tag):
    source = numeric_array(df, "n_src")
    background = numeric_array(df, "n_bkg")
    error = np.clip(numeric_array(df, "exp_angdist"), 0, 20)

    fig, ax = plt.subplots(figsize=(3.5, 2.8), constrained_layout=True)
    points = ax.scatter(
        source, background, c=error, cmap="viridis", vmin=0, vmax=20,
        marker=".", s=5, linewidths=0, rasterized=True,
    )
    colorbar = fig.colorbar(points, ax=ax)
    colorbar.set_label("Error (deg)")
    colorbar.set_ticks(
        [0, 5, 10, 15, 20], labels=["0", "5", "10", "15", "20+"]
    )
    ax.set_xlabel("Source event count $s$")
    ax.set_ylabel("Background event count $b$")
    ax.set_title(f"{tag.title()} Transients")
    return fig


for tag, frame in (("short", short_baseline_df), ("long", long_baseline_df)):
    figure = make_error_vs_events_figure(frame, tag)
    save_and_show(figure, f"map_err_vs_sb_{tag}")

## Figure 9

Impact of deadline on utility-based transient length determination relative to deadline-oblivious baseline, measured on long transient set for FoV 5.36 × 4.5°.

In [ ]:
def make_delta_length_figure(baseline_length, utility_length, deadline):
    delta = np.asarray(utility_length) - np.asarray(baseline_length)
    displayed = np.clip(delta, -10, 13)

    fig, ax = plt.subplots(figsize=(3.5, 2.7), constrained_layout=True)
    ax.hist(
        displayed,
        bins=np.linspace(-10.01, 12.99, num=12),
        density=True,
        align="left",
    )
    ax.set_xticks(
        [-10, -5, 0, 5, 10],
        [r"$\leq -10$", "-5", "0", "5", r"$\geq 10$"],
    )
    ax.set_xlabel("length(utility) - length(no-deadline) [s]")
    ax.set_ylabel("Fraction of transients")
    ax.set_title(f"Deadline $D={deadline}$ s")
    return fig, delta


# Loaded once and reused in both utility-comparison figures below.
long_reference_df = read_required_csv(
    LONG_BASELINE, {"est_length", "exp_angdist"}
)
utility_frames = {}

for deadline in (30, 60):
    utility_path = LONG_UTILITY_DIR / f"emsoft_stats_{deadline}.csv"
    utility_df = read_required_csv(
        utility_path, {"est_length", "exp_angdist"}
    )
    verify_paired_rows(long_reference_df, utility_df)
    utility_frames[deadline] = utility_df

    figure, delta = make_delta_length_figure(
        numeric_array(long_reference_df, "est_length"),
        numeric_array(utility_df, "est_length"),
        deadline,
    )
    delta_summary = pd.DataFrame([{
        "deadline_s": deadline,
        "n": len(delta),
        "fraction_shorter": np.mean(delta < 0),
        "fraction_unchanged": np.mean(delta == 0),
        "fraction_longer": np.mean(delta > 0),
        "median_change_s": np.median(delta),
    }])
    display(delta_summary.round({
        "fraction_shorter": 4, "fraction_unchanged": 4,
        "fraction_longer": 4, "median_change_s": 2,
    }))
    save_and_show(figure, f"hist_delta_length_{deadline}")

## Figure 10

Impact of deadline on mapping error of utility-based method, measured on long transient set for FoV 5.36 × 4.5°.
Transients are grouped by change in length vs. the deadline-oblivious method; x-axis labels show a representative value for each group. Within each group, a box plot shows the distribution of changes in error for the utility method vs. the baseline.

In [ ]:
def make_delta_error_figure(
    baseline_length,
    utility_length,
    baseline_error,
    utility_error,
    deadline,
    minimum_bin_size=50,
):
    delta_length = np.asarray(utility_length) - np.asarray(baseline_length)
    delta_error = np.asarray(utility_error) - np.asarray(baseline_error)

    order = np.argsort(delta_length)
    delta_length = delta_length[order]
    delta_error = delta_error[order]

    _, bin_edges = np.histogram(delta_length, bins=10)
    bin_edges[-1] = np.nextafter(bin_edges[-1], np.inf)
    bin_index = np.digitize(delta_length, bin_edges) - 1

    groups, labels, counts = [], [], []
    for index in np.unique(bin_index):
        values = delta_error[bin_index == index]
        if len(values) < minimum_bin_size:
            continue
        midpoint = 0.5 * (bin_edges[index] + bin_edges[index + 1])
        groups.append(values)
        labels.append(str(int(np.round(midpoint))))
        counts.append(len(values))

    if not groups:
        raise ValueError(
            f"No length-change bin contains at least {minimum_bin_size} rows"
        )

    fig, ax = plt.subplots(figsize=(3.5, 2.7), constrained_layout=True)
    draw_boxplot(ax, groups, labels)
    ax.set_xlabel("length(utility) - length(no-deadline) [s]")
    ax.set_ylabel("error(utility) - error(no-deadline) [deg]")
    ax.set_title(f"Deadline $D={deadline}$ s")
    return fig, pd.DataFrame({
        "representative_length_change_s": labels,
        "group_size": counts,
    })


for deadline, utility_df in utility_frames.items():
    figure, group_summary = make_delta_error_figure(
        numeric_array(long_reference_df, "est_length"),
        numeric_array(utility_df, "est_length"),
        numeric_array(long_reference_df, "exp_angdist"),
        numeric_array(utility_df, "exp_angdist"),
        deadline,
    )
    display(group_summary)
    save_and_show(figure, f"delta_err_vs_delta_length_{deadline}")

## Figure 11

Probability of successful transient detection by cooperating telescopes as a function of overall deadline D.
- (a) Short transients, FoV 5.36 × 4.5
- (b) Long transients, FoV 5.36 × 4.5
- (c) Short transients, FoV 2.5 × 2.5
- (d) Long transients, FoV 2.5 × 2.5


In [ ]:
METHOD_LABELS = {"utility": "Utility", "nodeadline": "No-Deadline"}
METHOD_COLORS = {"utility": "#3b7dd8", "nodeadline": "#d84b4b"}
METHOD_MARKERS = {"utility": "o", "nodeadline": "^"}


def compute_success_ratios(burst_tag, fov, deadlines, methods=("utility", "nodeadline")):
    records = []
    tiling = f"{fov}_tiling"
    result_directory = DATA_ROOT / "validation" / f"searching_results_{fov}"

    for method in methods:
        for deadline in deadlines:
            csv_path = (
                result_directory
                / f"{burst_tag}_{tiling}_{method}_{deadline}.csv"
            )
            frame = read_required_csv(csv_path, {"detected"})
            detected = detected_mask(frame["detected"])
            records.append({
                "method": method,
                "deadline_s": deadline,
                "n_success": int(detected.sum()),
                "n_total": len(detected),
                "success_probability": float(detected.mean()),
            })
    return pd.DataFrame(records)


def make_success_figure(summary):
    fig, ax = plt.subplots(figsize=(3.5, 2.0), constrained_layout=True)
    for method in ("utility", "nodeadline"):
        subset = summary.loc[summary["method"] == method].sort_values("deadline_s")
        ax.plot(
            subset["deadline_s"],
            subset["success_probability"],
            marker=METHOD_MARKERS[method],
            color=METHOD_COLORS[method],
            label=METHOD_LABELS[method],
            markersize=4,
            linewidth=1.2,
        )
    ax.set_xlabel("Overall Deadline (s)")
    ax.set_ylabel("Success Probability")
    ax.set_ylim(0.2, 1.02)
    ax.legend(loc="lower right")
    return fig


for paper_label, burst_tag, fov, deadlines in SUCCESS_CONFIGS:
    print(f"\n{paper_label}: burst={burst_tag}, FoV={fov}")
    success_summary = compute_success_ratios(burst_tag, fov, deadlines)
    display(success_summary.round({"success_probability": 4}))
    success_figure = make_success_figure(success_summary)
    save_and_show(
        success_figure,
        f"success_ratio_{burst_tag}_{fov}_tiling",
    )

## Optional: Reproduce Figure 4

Figure 4 reports platform-dependent mapping times. To reproduce it with newly measured results:

1. Run the applicable benchmarks in Section 4 of the artifact instructions.
2. Open the output files under:
   - `~/GRB-Search-Pipeline/results/dc3_benchmark/intel/`
   - `~/GRB-Search-Pipeline/results/dc3_benchmark/jetson/`
3. Find the reported time near the end of each file:

   ```text
   TIME: 0.402 s
4. Copy each numerical value into the corresponding time_means entry in the next cell.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

platforms = ("Intel", "Jetson")

# Fill in the mapping times obtained from the Section 4 benchmarks.
# Each tuple is ordered as: (Intel time, Jetson time), in seconds.
# Keep np.nan for COSIpy modes that cannot be run on the Jetson.

time_means = {
    "cosipy v3":    (None, np.nan),  # Intel: cosipy_interp
    "cosipy + JIT": (None, np.nan),  # Intel: cosipy_numba
    "ext-mem":      (None, None),    # emsoft_outmem
    "in-mem":       (None, None),    # emsoft_inmem
    "in-mem-mr":    (None, None),    # emsoft_moc
}

# Reminder
if any(
    value is None
    for measurements in time_means.values()
    for value in measurements
):
    raise ValueError(
        "Replace every None in time_means with the corresponding "
        "benchmark time reported by Section 4."
    )

x = len(time_means) * np.arange(len(platforms))
width = 0.8

fig, ax = plt.subplots()

for multiplier, (attribute, measurement) in enumerate(time_means.items()):
    offset = width * multiplier
    ax.bar(x + offset, measurement, width, label=attribute)

ax.set_ylabel("Mapping Time (s)")
ax.set_yscale("log")
ax.set_xticks(x + 2 * width, platforms)
ax.legend(loc="upper right", ncols=2)
ax.set_ylim(1e-1, 200)

output_path = OUTPUT_DIR / "barplot.pdf"
fig.savefig(output_path, bbox_inches="tight", pad_inches=0)
print(f"Saved {output_path}")
plt.show()
